In [50]:
# word2vecの高速か
# word2vecの改良

In [51]:
# Embeddingレイヤの実装
import numpy as np
W = np.arange(21).reshape(7, 3)
W

array([[ 0,  1,  2],
       [ 3,  4,  5],
       [ 6,  7,  8],
       [ 9, 10, 11],
       [12, 13, 14],
       [15, 16, 17],
       [18, 19, 20]])

In [52]:
idx = np.array([1, 0, 3, 0])
W[idx]

array([[ 3,  4,  5],
       [ 0,  1,  2],
       [ 9, 10, 11],
       [ 0,  1,  2]])

In [53]:
# Embaddingレイヤのforward()メソッドを実装
class Embedding:
    def __init__ (self, W):
        self.params = [W]
        self.grads = [np.zeros_like(W)]
        self.idx = None

    def forward(self, idx):
        W, = self.params
        self.idx = idx
        out =W[idx]
        return out

In [54]:
# Embaddingライやのbackward()メソッドの実装
def backward(self, dout):
    dW = self.grads
    dW[...] = 0
    
    for i, word_id in enumerate(self, idx):
        dW[word_id] += dout[i]

        return None

In [55]:
# word2vecの改良２
# 中間層以降の計算の問題点
# 多値分類から二値分類へ
# シグモイド関数と交差エントロピー誤差

In [56]:
# 多値分類から二値分類　実装
class EmbaddingDot:
    def __init__(self, W):
        self.embed = Embedding(W)
        self.params = self.embed.grads
        self.grads = self.embed.grads
        self.cache = None

    def forward(self, h, idx):
        target_W = self.embed.forward(idx)
        out = np.sum(target_W * h, axis=1)

        self.cache = (h, target_W)
        return out

    def backward(self, dout):
        h, target_W = self.cache
        dout = dout.reshape(dout.shape[0], 1)

        dtarget_W = dout * h
        dh = dout * target_W
        return dh

In [57]:
# Negative Sampling
# Negative Samplingのサンプリング手法

In [58]:
# 確率分布に従ったサンプリング
# np.random.choice()の例
import numpy as np

# 8~9の数字の中から1つの数字をランダムにサンプリング
np.random.choice(10)

0

In [59]:
# wordsから1つだけランダムにサンプリング
words = ["you", "say", "goodbye", "I", "hello", "."]
np.random.choice(words)

np.str_('goodbye')

In [60]:
# 5つだけランダムにサンプリング（重複あり）
np.random.choice(words, size=5)

array(['you', 'you', 'goodbye', 'I', 'goodbye'], dtype='<U7')

In [61]:
# 5つだけランダムサンプリング（重複なし）
np.random.choice(words, size=5, replace=False)

array(['you', 'say', 'I', 'goodbye', '.'], dtype='<U7')

In [62]:
# 確率分布に従ってサンプリング
p = [0.5, 0.1, 0.05, 0.2, 0.05, 0.1]
np.random.choice(words, p=p)

np.str_('hello')

In [63]:
p = [0.7, 0.29, 0.01]
new_p = np.power(p, 0.75)
new_p /= np.sum(new_p)
print(new_p)

[0.64196878 0.33150408 0.02652714]


In [64]:
# UnigraSamplerクラスの例
from negative_sampling_layer import UnigramSampler


corpus = np.array([0, 1, 2, 3, 4, 1, 2, 3])
power = 0.75
sample_size = 2

sampler = UnigramSampler(corpus, power, sample_size)
target = np.array([1, 3, 0])
negative_sample = sampler.get_negative_sample(target)
print(negative_sample)

[[3 2]
 [0 4]
 [3 2]]


In [65]:
# Negative Samplingの実装
from common.layers import SigmoidWithLoss
from negative_sampling_layer import UnigramSampler, EmbeddingDot

class NegativeSamplingLoss:
    def __init__(self, W, corpus, power=0.75, sample_size=5):
        self.sample_size = sample_size
        self.sampler = UnigramSampler(corpus, power, sample_size)
        self.loss_layers = [SigmoidWithLoss() for _ in range(sample_size + 1)]
        self.embed_dot_layers = [EmbeddingDot(W) for _ in range(sample_size + 1)]

        self.params, self.grads = [], []
        for layer in self.embed_dot_layers:
            self.params += layer.params
            self.grads += layer.grads

In [66]:
# 順伝播の実装
def forward(self, h, target):
    batch_size = target.shape[0]
    negative_sample = self.sampler.get_negative_sample(target)

    # 正例のフォワード
    score = self.embed_dot_layers[0].forward(h, target)
    correct_label = np.ones(batch_size, dtype=np.int32)
    loss = self.loss_layers[0].forward(score, correct_label)

    # 負例のフォワード
    negative_label = np.zeros(batch_size, dtype=np.int32)
    for i in range(self.sample_size):
        negative_target = negative_sample[:, 1]
        score = self.embed_dot_layers[1 + i],forward(h, negative_target)
        loss += self.loss_layers[1 + i].forward(score, negative_label)

    return loss

In [67]:
# 逆伝播の実装
def backward(self, dout):
    dh = 0
    for l0, l1 in zip(self.loss_layers, self.embed_dot_layers):
        dscore = l0.backward(dout)
        dh += l1.backward(dscore)
    return dh

In [75]:
class NegativeSamplingLoss:
    def __init__(self, W, corpus, power=0.75, sample_size=5):
        self.sample_size = sample_size
        self.sampler = UnigramSampler(corpus, power, sample_size)
        self.loss_layers = [SigmoidWithLoss() for _ in range(sample_size + 1)]
        self.embed_dot_layers = [EmbeddingDot(W) for _ in range(sample_size + 1)]

        self.params, self.grads = [], []
        for layer in self.embed_dot_layers:
            self.params += layer.params
            self.grads += layer.grads

    def forward(self, h, target):
        batch_size = target.shape[0]
        negative_sample = self.sampler.get_negative_sample(target)

        # 正例のフォワード
        score = self.embed_dot_layers[0].forward(h, target)
        correct_label = np.ones(batch_size, dtype=np.int32)
        loss = self.loss_layers[0].forward(score, correct_label)

        # 負例のフォワード
        negative_label = np.zeros(batch_size, dtype=np.int32)
        for i in range(self.sample_size):
            negative_target = negative_sample[:, i]
            score = self.embed_dot_layers[1 + i].forward(h, negative_target)
            loss += self.loss_layers[1 + i].forward(score, negative_label)

        return loss

    def backward(self, dout=1):
        dh = 0
        for l0, l1 in zip(self.loss_layers, self.embed_dot_layers):
            dscore = l0.backward(dout)
            dh += l1.backward(dscore)

        return dh

In [ ]:
# 改良版word2vecの学習
# CBOWモデルの実装
import sys
from typing import get_overloads
sys.path.append("..")
import numpy as np
from common.layers import Embedding

rng = np.random.default_rng()

class CBOW:
    def __init__(self, vocab_size, hidden_size, window_size, corpus):
        V, H = vocab_size, hidden_size

        # 重みの初期化
        W_in = 0.01 + rng.random((V, H)).astype("f")
        W_out = 0.01 + rng.random((V, H)).astype("f")

        # レイヤの生成
        self.in_layers = []
        for i in range(2 * window_size):
            layer = Embedding(W_in) # Embeddingレイヤを使用
            self.in_layers.append(layer)
        self.ns_loss = NegativeSamplingLoss(W_out, corpus, power=0.75, sample_size=5)

        # 全ての重みと勾配を配列にまとめる
        layers = self.in_layers + [self.ns_loss]
        self.params, self.grads = [], []
        for layer in layers:
            self.params += layer.params
            self.grads += layer.grads
            
        # メンバ変数に単語の分散表現を設定
        self.word_vecs =W_in

In [ ]:
# forward()メソッドとbackward()メソッドを実装
def forward(self, contexts, target):
    h = 0
    for i, layer in enumerate(self.in_layers):
        h += layer.forward(contexts[:, i])
    h += 1 / len(self.in_layers)
    loss = self.ns_loss.forward(h, target)
    return loss

def backward(self, dout=1):
    dout = self.ns_loss.backward(dout)
    dout += 1 / len(self.in_layers)
    for layer in self.in_layers:
        layer.backward(dout)
    return None